In [1]:
from faster_whisper import WhisperModel


In [6]:
def transcribe_audio(file_path: str, model_size: str = "tiny") -> str:
    """
    Transcribe audio using the FasterWhisper package.

    Args:
        file_path (str): Path to the input file (or a file-like object), or the audio waveform.
        model_size (str): Size of the model to use (tiny, tiny.en, base, base.en,
            small, small.en, distil-small.en, medium, medium.en, distil-medium.en, large-v1,
            large-v2, large-v3, large, distil-large-v2 or distil-large-v3), a path to a
            converted model directory, or a CTranslate2-converted Whisper model ID from the HF Hub.
            When a size or a model ID is configured, the converted model is downloaded
            from the Hugging Face Hub.

    Returns:
        transcription (str): transcribed audio in string format for the provided input audio or video. 
    """

    # Initalize WhisperModel as shown on fasterwhisper instructions. Use the adequate arguments.
    model = WhisperModel(model_size_or_path=model_size, 
                         device='auto', 
                         compute_type="float32")

    # Perform the transcription 
    segments, _ = model.transcribe(file_path)

    # Define object to return transcription
    transcription = ''

    # Merge all segments composing the transcription
    for i, segment in enumerate(segments):
        
        text = segment.text

        # Eliminate spaces at the beggining of the text in the first segment
        if i == 0 and segment.text[0] == ' ':
            while text[0] == ' ':
                text = text[1:]
        else:
            text = segment.text

        transcription += text

    # Format text setting the first letter as capital
    transcription = transcription.capitalize()

    return transcription


In [ ]:
filepath = ""

transcription = transcribe_audio(filepath, )

In [3]:
FILEPATH1 = 'audio1.ogg'
FILEPATH2 = 'audio2.ogg'


In [8]:
transcriptions = {}

for model_size in ['tiny', 'large']:
    if not model_size in transcriptions.keys():
        transcriptions[model_size] = {}

    for audio in [FILEPATH1, FILEPATH2]:
        if not audio in transcriptions[model_size].keys():
            transcriptions[model_size][audio] = transcribe_audio(audio, model_size)


config.json:   0%|          | 0.00/2.39k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

vocabulary.json:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [9]:
transcriptions

{'tiny': {'audio1.ogg': 'Robota avanzas 3 metros, gira la derecha 90º, se estén en el verazo 30 a 5 metros y sujeta el objeto con fuerza media.',
  'audio2.ogg': 'Robot, avanzas tres metros, gira la derecha 90 grados, extiende las 30 centímetros y sujeta el objeto con fuerza media.'},
 'large': {'audio1.ogg': 'Robot avanza 3 metros, gira a la derecha 90 grados, extiende el brazo 30 centímetros y sujeta el objeto con fuerza media.',
  'audio2.ogg': 'Robot, avanza 3 metros, gira a la derecha 90 grados, extiende el brazo 30 centímetros y sujeta el objeto con fuerza media.'}}

In [10]:
from jiwer import wer


In [2]:
ground_truth = '' # Define the ground truth of the audio


In [12]:
for model_size in ['tiny', 'large']:
    for audio in [FILEPATH1, FILEPATH2]:
        transcription_ = transcriptions[model_size][audio]

        error = wer(ground_truth, transcription_)
        print(f'MS: {model_size}, Audio: {audio}, Word Error Rate: {error:.2%}')
        

MS: tiny, Audio: audio1.ogg, Word Error Rate: 59.09%
MS: tiny, Audio: audio2.ogg, Word Error Rate: 27.27%
MS: large, Audio: audio1.ogg, Word Error Rate: 18.18%
MS: large, Audio: audio2.ogg, Word Error Rate: 13.64%
